In [3]:
!pip install numpy nltk torch google-generativeai sentence-transformers

  Using cached numpy-2.4.4-cp313-cp313-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached torch-2.11.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (29 kB)
  Using cached sentence_transformers-5.4.1-py3-none-any.whl.metadata (17 kB)
  Using cached click-8.3.3-py3-none-any.whl.metadata (2.6 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached requests-2.33.1-py3-none-any.whl.metadata (4.8 kB)
INFO: pip is looking at multiple versions of grpc

In [6]:
import json
import numpy as np
import nltk
import torch
import google.generativeai as genai
from sentence_transformers import CrossEncoder
from pathlib import Path

In [8]:
KB_PATH = "./knowledge_base.json"

In [10]:
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/ananyaravichandran/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/ananyaravichandran/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [11]:
!pip install -U -q google-generativeai


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import os

GEMINI_KEY = os.getenv("GEMINI_KEY")
genai.configure(api_key=GEMINI_KEY)
model = genai.GenerativeModel("gemini-3-flash-preview")

DRIVE_BASE = Path("/content/drive/MyDrive/CS_510")
KB_PATH = DRIVE_BASE / "knowledge_base.json"
OUTPUT_JSON = DRIVE_BASE / "claim_vs_reality_final.json"

# Ranking Model
print("Loading Cross-Encoder for ranking...")
device = "cuda" if torch.cuda.is_available() else "cpu"
ranker = CrossEncoder('cross-encoder/stsb-distilroberta-base', device=device)

with open(KB_PATH, "r", encoding="utf-8") as f:
    kb = json.load(f)

def ask_gemini(prompt):
    """Helper to get concise responses from Gemini."""
    try:
        response = model.generate_content(prompt)
        return response.text.strip().replace("*", "")
    except Exception as e:
        print(f"Gemini Error: {e}")
        return ""

def generate_smart_claim_vs_reality(kb, test_mode=True):
    all_products_output = []

    for product_name, entry in kb.items():
        description = entry.get("combined_description", "")
        reviews = [r.get("body", "") for r in entry.get("all_reviews", [])]

        if not description or not reviews:
            continue

        print(f"Processing: {product_name}")

        claim_sentences = nltk.sent_tokenize(description)
        review_sentences = []
        for r_text in reviews:
            review_sentences.extend(nltk.sent_tokenize(r_text))

        if not review_sentences:
            continue

        claim_vs_reality_rows = []

        for c_sent in claim_sentences[:3]:
            label_prompt = f"Convert this product claim into a 2-3 word title (e.g., 'Fit and comfort'). Claim: {c_sent}"
            feature_label = ask_gemini(label_prompt).title()


            pairs = [[c_sent, r_sent] for r_sent in review_sentences]
            scores = ranker.predict(pairs)


            top_indices = np.argsort(scores)[-5:][::-1]
            top_relevant_sentences = [review_sentences[i] for i in top_indices if scores[i] > 0.15]

            if not top_relevant_sentences:
                actual_summary = "No significant user feedback found."
            else:
                context = " ".join(top_relevant_sentences)
                summary_prompt = f"Summarize these user reviews about '{feature_label}' into one concise sentence: {context}"
                actual_summary = ask_gemini(summary_prompt)

            claim_vs_reality_rows.append({
                "feature": feature_label or "Product Feature",
                "advertised": c_sent,
                "actual": actual_summary
            })

        product_data = {
            "product_name": product_name,
            "claim_vs_reality": claim_vs_reality_rows
        }

        all_products_output.append(product_data)

        if test_mode:
            print("\n--- TEST MODE OUTPUT ---")
            print(json.dumps(product_data, indent=2))
            break

    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(all_products_output, f, indent=2, ensure_ascii=False)

    print(f"\n Final formatted JSON saved to: {OUTPUT_JSON}")
    return all_products_output

output_data = generate_smart_claim_vs_reality(kb, test_mode=False)

NameError: name 'userdata' is not defined